# Notebook 3 — AI-Assisted Research & Development with DeepSeek

**Workshop:** Edge Intelligence, Bio-Inspired Optimization & UAV-Assisted VANETs — ITS IRM 2026
**Estimated time:** 60–75 minutes, fully self-paced
**Requires:** a free chat.deepseek.com account. **No API key, no billing, no installation.**
**Builds on:** your own working code from Notebook 2 (Exercise C reuses it directly)

### How this notebook works
There is no automatic connection to DeepSeek here — you copy a prompt out of this notebook, paste it
into a **chat.deepseek.com** browser tab, and paste DeepSeek's answer back in. That's the entire
interaction model, and it's deliberate: it needs no account setup beyond a free chat login, no API
billing, and no facilitator involvement to get running.

What *is* automated is the **verification** — every exercise below has a self-check cell that tells you
`PASSED` or exactly what's wrong, the same as Notebooks 1 and 2.

| Exercise | Skill | Verified by |
|---|---|---|
| A — Debugging | Using an LLM to fix real code | Automated test cases (assert-based) |
| B — Literature synthesis | Using an LLM for lit review without hallucinating citations | An automated quote-checker you run on DeepSeek's actual text |
| C — Algorithm design | Brainstorming with an LLM, then implementing and benchmarking it yourself | Your own Grey Wolf clustering code from Notebook 2 |

### One-time setup
Go to **chat.deepseek.com** now and create a free account (email or phone). Keep that tab open —
you'll come back to it three times below. Nothing else to install or configure.

### Setup — run this first (no login, no key, just imports)

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt

print("Setup complete. No DeepSeek account or key needed for this cell — just switch to your")
print("chat.deepseek.com browser tab whenever a 'COPY THIS PROMPT' block appears below.")

## Exercise A — DeepSeek-Assisted Debugging (20 min)

The function below computes **cluster-head stability** — the fraction of rounds where the same
cluster-head set persisted from the previous round (the same concept you measured in Notebook 2).
It has a real, specific bug.

In [ ]:
def compute_cluster_stability_BUGGY(head_history):
    """head_history: list of sets, one per round, containing cluster-head node ids for that round.
    Returns the fraction of rounds where the head set matched the previous round."""
    same_count = 0
    for i in range(len(head_history)):
        if head_history[i] == head_history[i - 1]:
            same_count += 1
    return same_count / len(head_history)

test_history = [{1, 2, 3}, {1, 2, 3}, {4, 5, 6}, {4, 5, 6}, {4, 5, 6}]
print("Buggy result:", compute_cluster_stability_BUGGY(test_history))
print("(By hand: rounds 1-4 compared to the previous round give 3 matches out of 4 comparisons = 0.75)")

**Step 1.** Run the next cell — it prints a ready-to-paste prompt (no DeepSeek needed for this step).

In [ ]:
prompt_A = """This Python function should return the fraction of rounds where the cluster-head
set matches the PREVIOUS round: round 1 compared to round 0, round 2 compared to round 1, and so on.
The very first round has no previous round to compare against, so it must be excluded from both the
count and the denominator. The function currently gives an incorrect result. Find the bug, fix it,
and explain what was wrong in plain language.

def compute_cluster_stability_BUGGY(head_history):
    same_count = 0
    for i in range(len(head_history)):
        if head_history[i] == head_history[i - 1]:
            same_count += 1
    return same_count / len(head_history)
"""
print("=" * 60)
print("COPY EVERYTHING BELOW THIS LINE INTO chat.deepseek.com")
print("=" * 60)
print(prompt_A)

**Step 2.** Paste the copied text into chat.deepseek.com and send it. Read its explanation, then copy
its corrected function into the cell below, replacing `pass`.

In [ ]:
# PASTE_HERE: replace `pass` with DeepSeek's fixed function
def compute_cluster_stability_FIXED(head_history):
    # PASTE DEEPSEEK'S FIXED VERSION HERE
    pass

### Self-check — run this to verify the fix

In [ ]:
_passed = True
_result = compute_cluster_stability_FIXED(test_history)
if _result is None:
    print("compute_cluster_stability_FIXED returned None — did DeepSeek's fix get pasted in correctly?")
    _passed = False
elif abs(_result - 0.75) > 1e-9:
    print(f"Expected 0.75, got {_result}. Try asking DeepSeek to double-check the loop's start")
    print("index and the denominator.")
    _passed = False

# a second, independent test case, so a fix that "works" by coincidence still gets caught
test_history_2 = [{1}, {2}, {2}, {2}, {1}, {1}]  # comparisons: F, T, T, F, T -> 3/5 = 0.6
_result_2 = compute_cluster_stability_FIXED(test_history_2)
if _result_2 is not None and abs(_result_2 - 0.6) > 1e-9:
    print(f"Second test case failed: expected 0.6, got {_result_2}.")
    _passed = False

if _passed:
    print("Exercise A self-check: ALL TESTS PASSED ✅")
    print("DeepSeek's fix (as you applied it) is verified correct against two independent test cases.")
assert _passed

**Discussion (no code needed):** Did DeepSeek's explanation correctly diagnose *why* the original code
was wrong, or did it just patch the symptom? What would you have had to check even if the fix had
looked plausible on first read?

## Exercise B — Literature Synthesis Without Hallucinating (15 min)

You'll ask DeepSeek to connect three real abstracts, then **automatically check every quote it gives
you** against the actual source text. This is the core academic-integrity skill for using an LLM in a
literature review.

In [ ]:
abstracts = {
    "J4 (CAMONET)": (
        "Vehicular ad-hoc networks require efficient clustering to reduce control overhead in highly "
        "dynamic topologies. This paper proposes CAMONET, a Moth-Flame Optimization based clustering "
        "scheme that elects stable cluster heads by modeling candidate selection as moths spiraling "
        "toward a flame. Simulation results show improved cluster stability and reduced control "
        "overhead compared to classical probabilistic clustering."
    ),
    "J34 (Harris Hawks)": (
        "This paper presents a Harris Hawks Optimization based clustering algorithm for vehicular "
        "ad-hoc networks. Inspired by the cooperative surprise-pounce hunting strategy of Harris hawks, "
        "the proposed method converges on cluster-head candidates from multiple directions "
        "simultaneously. Results on standard VANET mobility traces show superior cluster-head stability "
        "and lower re-election frequency compared to Grey Wolf and Moth-Flame based approaches."
    ),
    "J16 (UAV-VANET)": (
        "Urban vehicular networks suffer coverage gaps due to building obstructions and roadside unit "
        "density limits. This paper proposes a UAV-assisted communication architecture in which an "
        "aerial relay node dynamically repositions toward regions of high vehicle density, extending "
        "effective network coverage. Results show improved packet delivery ratio and reduced end-to-end "
        "delay compared to a roadside-unit-only baseline."
    ),
}
for k, v in abstracts.items():
    print(f"[{k}]\n{v}\n")

In [ ]:
prompt_B = """Given these three paper abstracts, identify ONE open research gap that connects all
three. For every claim you make, state exactly which abstract supports it and quote the exact
supporting phrase in double quotes.

[J4 (CAMONET)] """ + abstracts["J4 (CAMONET)"] + """

[J34 (Harris Hawks)] """ + abstracts["J34 (Harris Hawks)"] + """

[J16 (UAV-VANET)] """ + abstracts["J16 (UAV-VANET)"] + """
"""
print("=" * 60)
print("COPY EVERYTHING BELOW THIS LINE INTO chat.deepseek.com")
print("=" * 60)
print(prompt_B)

**Step 2.** Paste DeepSeek's *entire* response into the cell below, between the triple quotes,
keeping its own quotation marks exactly as it wrote them.

In [ ]:
# PASTE_HERE: DeepSeek's full response, quotation marks intact
deepseek_synthesis = """PASTE DEEPSEEK'S FULL RESPONSE HERE"""

### Automated hallucination check — run this on DeepSeek's actual response

In [ ]:
def check_quotes_against_sources(llm_response, source_texts):
    combined = re.sub(r"\s+", " ", " ".join(source_texts).lower())
    quotes = re.findall(r'"([^"]{8,})"', llm_response)
    return [(q, re.sub(r"\s+", " ", q.strip().lower()) in combined) for q in quotes]

_results = check_quotes_against_sources(deepseek_synthesis, list(abstracts.values()))

if not _results:
    print("No quoted phrases (in \"double quotes\") were found in your pasted response.")
    print("Make sure you pasted DeepSeek's actual answer above (not the placeholder text),")
    print("and that it used double quotation marks. If it didn't, re-send the prompt with:")
    print("  'Format every supporting quote in double quotation marks.'")
else:
    for q, found in _results:
        status = "VERIFIED ✅" if found else "⚠️  NOT FOUND IN SOURCE — possible hallucination"
        print(f"[{status}] \"{q}\"")
    n_verified = sum(f for _, f in _results)
    print(f"\n{n_verified}/{len(_results)} quoted claims verified against the actual abstracts.")
    if n_verified < len(_results):
        print("Any flagged quote should NOT be cited in real work without manually re-checking it —")
        print("this is expected and is the whole point of the exercise, not a failure.")

**Discussion (no code needed):** A "VERIFIED" quote being present verbatim doesn't guarantee DeepSeek
used it *in context correctly* — read the surrounding sentence in the abstract yourself. What's the
difference between "this phrase appears in the source" and "this claim is actually true"?

## Exercise C — LLM-Assisted Hybrid-Algorithm Design & Benchmarking (25 min)

This exercise **directly reuses your own work from Notebook 2**. Use DeepSeek as a brainstorming
partner, then implement and benchmark the idea yourself.

### Step 1 — Bring in your Grey Wolf update function from Notebook 2

Copy your own `gwo_update_position` function — the one that passed Notebook 2's Exercise 1 self-check —
and paste it into the cell below, replacing the placeholder. If you'd rather not go back to Notebook 2,
a working reference version is given in a comment you can uncomment instead.

In [ ]:
# PASTE_HERE: your own gwo_update_position from Notebook 2 (the one that passed its self-check)
def gwo_update_position(X, X_alpha, X_beta, X_delta, a, r):
    r1a, r2a, r1b, r2b, r1d, r2d = r
    # PASTE YOUR NOTEBOOK 2 IMPLEMENTATION HERE
    pass

# --- Didn't finish Notebook 2, or want to double-check your version? Uncomment the block below ---
# def gwo_update_position(X, X_alpha, X_beta, X_delta, a, r):
#     r1a, r2a, r1b, r2b, r1d, r2d = r
#     A1, C1 = 2*a*r1a - a, 2*r2a
#     X1 = X_alpha - A1 * np.abs(C1*X_alpha - X)
#     A2, C2 = 2*a*r1b - a, 2*r2b
#     X2 = X_beta - A2 * np.abs(C2*X_beta - X)
#     A3, C3 = 2*a*r1d - a, 2*r2d
#     X3 = X_delta - A3 * np.abs(C3*X_delta - X)
#     return (X1 + X2 + X3) / 3.0

### Quick check — confirm your update function is wired up correctly before continuing

In [ ]:
_rng = np.random.default_rng(7)
_X, _Xa, _Xb, _Xd = (_rng.uniform(0, 10, size=12) for _ in range(4))
_a = 1.3
_r = tuple(_rng.uniform(0, 1, size=12) for _ in range(6))
_r1a, _r2a, _r1b, _r2b, _r1d, _r2d = _r
_expected = ((_Xa - (2*_a*_r1a - _a) * np.abs(2*_r2a*_Xa - _X))
           + (_Xb - (2*_a*_r1b - _a) * np.abs(2*_r2b*_Xb - _X))
           + (_Xd - (2*_a*_r1d - _a) * np.abs(2*_r2d*_Xd - _X))) / 3.0
_got = gwo_update_position(_X, _Xa, _Xb, _Xd, _a, _r)
_passed = _got is not None and np.allclose(_expected, _got, atol=1e-9)
if _passed:
    print("gwo_update_position wired up correctly ✅ — continue to Step 2.")
else:
    print("gwo_update_position isn't matching the reference equation yet.")
    print("Paste your working Notebook 2 version, or uncomment the reference block above.")
assert _passed

### Step 2 — Given: the clustering benchmark (same as Notebook 2, provided so this notebook is self-contained)

In [ ]:
def make_nodes(n, seed):
    rng = np.random.default_rng(seed)
    return rng.uniform(0, 1000, size=(n, 2))

def fitness(flat_centroids, nodes, k):
    centroids = flat_centroids.reshape(k, 2)
    d = np.linalg.norm(nodes[:, None, :] - centroids[None, :, :], axis=2)
    return d.min(axis=1).sum()

def run_gwo(nodes, k, iterations, pop_size, seed):
    rng = np.random.default_rng(seed)
    dim = k * 2
    lo, hi = 0, 1000
    wolves = rng.uniform(lo, hi, size=(pop_size, dim))
    fitnesses = np.array([fitness(w, nodes, k) for w in wolves])
    order = np.argsort(fitnesses)
    X_alpha, X_beta, X_delta = wolves[order[0]], wolves[order[1]], wolves[order[2]]
    history = [fitnesses[order[0]]]
    for it in range(iterations):
        a = 2 - it * (2 / iterations)
        for i in range(pop_size):
            r = tuple(rng.uniform(0, 1, size=dim) for _ in range(6))
            wolves[i] = np.clip(gwo_update_position(wolves[i], X_alpha, X_beta, X_delta, a, r), lo, hi)
        fitnesses = np.array([fitness(w, nodes, k) for w in wolves])
        order = np.argsort(fitnesses)
        X_alpha, X_beta, X_delta = wolves[order[0]], wolves[order[1]], wolves[order[2]]
        history.append(fitnesses[order[0]])
    return X_alpha, history

N_SNAPSHOTS, K = 15, 6
gwo_results = []
for seed in range(N_SNAPSHOTS):
    nodes = make_nodes(60, seed=200 + seed)
    _, history = run_gwo(nodes, K, iterations=40, pop_size=15, seed=seed)
    gwo_results.append(history[-1])
print(f"Your plain-GWO baseline (avg. over {N_SNAPSHOTS} snapshots): {np.mean(gwo_results):.1f}")

### Step 3 — Ask DeepSeek for a hybrid idea

In [ ]:
prompt_C = """I have two VANET clustering algorithms:
1) Grey Wolf Optimization - pack hierarchy (alpha/beta/delta/omega) guides candidate solutions
   toward the best-known cluster-head positions.
2) Harris Hawks Optimization - a cooperative "surprise pounce" strategy converges on cluster heads
   from multiple directions simultaneously.

Propose ONE concrete, implementable way to hybridize these two for VANET clustering. Explain the
expected trade-off versus each algorithm alone, and outline the idea in pseudocode using no more
than 15 lines."""
print("=" * 60)
print("COPY EVERYTHING BELOW THIS LINE INTO chat.deepseek.com")
print("=" * 60)
print(prompt_C)

**Step 4.** Read DeepSeek's suggestion, then implement a minimal version of it below — aim for a
10–15 line change to `run_gwo`, not a rewrite. Paste DeepSeek's idea into the docstring first, for
your own record.

In [ ]:
def run_hybrid(nodes, k, iterations, pop_size, seed):
    """
    PASTE A ONE-LINE SUMMARY OF DEEPSEEK'S SUGGESTED HYBRID HERE.
    """
    # YOUR IMPLEMENTATION HERE — start by copying run_gwo above, then apply DeepSeek's idea
    pass

### Benchmark your hybrid against your Notebook-2-equivalent baseline above

In [ ]:
hybrid_results = []
try:
    for seed in range(N_SNAPSHOTS):
        nodes = make_nodes(60, seed=200 + seed)
        result = run_hybrid(nodes, K, iterations=40, pop_size=15, seed=seed)
        if result is None:
            raise ValueError("run_hybrid returned None — did you finish implementing it?")
        _, history = result
        hybrid_results.append(history[-1])

    print(f"Plain GWO average:   {np.mean(gwo_results):.1f}")
    print(f"Hybrid average:      {np.mean(hybrid_results):.1f}")
    improvement = 100 * (np.mean(gwo_results) - np.mean(hybrid_results)) / np.mean(gwo_results)
    print(f"\nHybrid vs. plain GWO: {improvement:+.1f}% (positive = hybrid is better)")

    plt.figure(figsize=(6, 4.5))
    plt.bar(["Plain GWO", "Your Hybrid"], [np.mean(gwo_results), np.mean(hybrid_results)],
            color=["#DCE6EC", "#065A82"])
    plt.ylabel("Avg. intra-cluster distance (lower = better)")
    plt.title(f"Your hybrid vs. plain GWO across {N_SNAPSHOTS} snapshots")
    plt.show()
except Exception as e:
    print("Not runnable yet:", e)
    print("Finish the run_hybrid implementation above, then re-run this cell.")

### There is no "must pass" assertion here — and that's intentional

A result showing your hybrid **did not** beat plain GWO is a completely valid, reportable finding —
record it and your hypothesis for why, exactly as you would in a real research log. Bring your number
(better or worse) to the afternoon showcase, alongside your Notebook 1 chart and Notebook 2 bar chart.

---
## Responsible LLM Use in Research — keep these in mind going forward

- Always verify factual claims, especially citations, dates, and numbers, against the primary source
- Disclose AI assistance according to your institution's and your target journal's policy
- Never paste unpublished results or confidential data into a public LLM chat interface
- Use LLMs for first drafts, brainstorming, and debugging — not as the final word on correctness
- Keep a record of prompts for anything that ends up influencing a publication

## Prompt Cheat Sheet (for your own future use)

| Task | Prompt Template |
|---|---|
| Debug code | "This function should do X but produces Y. Find the bug, fix it, and explain what was wrong: [code]" |
| Literature synthesis | "Given these abstracts, find one connecting gap. Cite which abstract supports each claim: [abstracts]" |
| Algorithm brainstorming | "I have algorithms A and B, which do X and Y. Propose one concrete way to hybridize them for [problem], with pseudocode." |

---
**You're done.** Bring your three results to the project showcase.